# Final project E+ model results generation

In [ ]:
import pandas as pd
import numpy as np
import eppy
from eppy.modeleditor import IDF
from eppy import modeleditor
import esoreader
import os
import sys
from itertools import product
from tqdm import trange
import shutil
import matplotlib.pyplot as plt

In [ ]:
def clear_output():
    """
    clear output for both jupyter notebook and the console
    """
    os.system('cls' if os.name == 'nt' else 'clear')
    if 'ipykernel' in sys.modules:
        from IPython.display import clear_output as clear
        clear()

In [ ]:
root_dir = os.getcwd()

## Run simulation

In [ ]:
eplus_root = r"D:\EnergyPlusV25-1-0" # replace with your EnergyPlus installation path
iddfile = os.path.join(eplus_root,"Energy+.idd")
try:
    IDF.setiddname(iddfile)
except modeleditor.IDDAlreadySetError as e:
    pass

In [ ]:
class ESO:
    def __init__(self, path):
        self.dd, self.data = esoreader.read(path)
    def read_var(self, variable, frequency = "Hourly"):
        return [
            {"key": k,
             "series": self.data[self.dd.index[frequency, k, variable]]}
            for _f, k, _v in self.dd.find_variable(variable)
        ]
    def get_df(self, variable, frequency = "Hourly"):
        dic = self.read_var(variable, frequency)
        key = [each["key"] for each in dic]
        values = [each["series"] for each in dic]
        df = pd.DataFrame(values,index = key).T
        return df
    def total_kwh(self, variable, frequency = "Hourly"):
        j_per_kwh = 3_600_000
        results = self.read_var(variable,frequency)
        return sum(sum(s["series"]) for s in results)/j_per_kwh

In [ ]:
idf = IDF(os.path.join(root_dir,r"model_week4\model_week4.idf")) # replace with your path

unit transformation

In [ ]:
new_wall_Rvalue = 14.3 * 0.1761108  # convert to m2K/W
new_SHGC = 0.25
new_occupancy_density = 0.04
new_cooling_setpoint = 26

In [ ]:
Location = ["San Francisco","Sacramento","Chicago","New York"]
Wall_Rvalue = [14.3,21.9,29.7]  # in ft2Fhr/Btu
WWR = [0.25,0.40,0.60]
SHGC = [0.25,0.40,0.60]
Occupancy_Density = [0.04,0.06,0.08]  # in person/m2
Lights_Density = [7.5,9]  # in W/m2
Cooling_Setpoint = [24,26,28]  # in C
keys = ["Location","Wall_Rvalue","WWR","SHGC","Occupancy_Density","Lights_Density","Cooling_Setpoint"]
dic = [dict(zip(keys, v)) for v in product(Location,Wall_Rvalue,WWR,SHGC,Occupancy_Density,Lights_Density,Cooling_Setpoint)]
df = pd.DataFrame(dic)
df["Wall_Rvalue"] = df["Wall_Rvalue"] * 0.1761108  # convert to m2K/W

In [ ]:
# replace with your weather file paths
MAP_WEATHER = {
    "San Francisco":os.path.join(root_dir,r"weather_data\USA_CA_San.Francisco.Intl.AP.724940_TMY3.epw"),
    "Sacramento":os.path.join(root_dir,r"weather_data\USA_CA_Sacramento.Exec.AP.724830_TMY3.epw"),
    "Chicago":os.path.join(root_dir,r"weather_data\USA_IL_Chicago-OHare.Intl.AP.725300_TMY3.epw"),
    "New York":os.path.join(root_dir,r"weather_data\USA_NY_New.York-J.F.Kennedy.Intl.AP.744860_TMY3.epw"),
}

In [ ]:
MAP_WWR = {
    0.25: [2,3.75], 
    0.40: [2,6],
    0.60: [0.5,9], #(10*3)*0.6/2
}

In [ ]:
def modify_idf(idf,modify_dict):
    material = idf.idfobjects["MATERIAL"]
    new_wall_k = (1/modify_dict["Wall_Rvalue"])*material[-1]["Thickness"]
    material[-1]["Conductivity"] = new_wall_k

    window = idf.idfobjects["WINDOW"]
    new_window_x, new_window_length = MAP_WWR[modify_dict["WWR"]]
    window[0]["Starting_X_Coordinate"] = new_window_x
    window[0]["Length"] = new_window_length

    window_material = idf.idfobjects["WINDOWMATERIAL:SIMPLEGLAZINGSYSTEM"]
    window_material[0]["Solar_Heat_Gain_Coefficient"] = modify_dict["SHGC"]

    occupancy = idf.idfobjects["PEOPLE"]
    occupancy[0]["Number_of_People_Calculation_Method"] = "People/Area"
    occupancy[0]["People_per_Floor_Area"] = modify_dict["Occupancy_Density"]

    light = idf.idfobjects["LIGHTS"]
    light[0]["Watts_per_Floor_Area"] = modify_dict["Lights_Density"]

    schedule = idf.idfobjects["SCHEDULE:COMPACT"]
    schedule[0]["Field_6"] = 12 #turn off heating
    schedule[1]["Field_6"] = modify_dict["Cooling_Setpoint"]

    weather_file = MAP_WEATHER[modify_dict["Location"]]
    idf.epw = weather_file
    return idf

In [ ]:
base_path = os.path.join(root_dir, "final_project_ep_results") 
if not os.path.exists(base_path):
    os.mkdir(base_path)

In [ ]:
os.chdir(root_dir)
result_df = pd.DataFrame(columns=["Total_Cooling_kwh"])
for groupname in trange(len(df)):
    modify_dict = df.iloc[groupname].to_dict()
    idf = IDF(os.path.join(root_dir,r"model_week4\model_week4.idf")) # replace with your path
    idf = modify_idf(idf,modify_dict)
    try:
        os.mkdir(os.path.join(base_path,f"{groupname}") )   
    except:
        pass
    os.chdir(os.path.join(base_path,f"{groupname}"))
    idf.run(expandobjects=True)
    eso = ESO(os.path.join(base_path,f"{groupname}\\eplusout.eso"))
    result_df.loc[groupname] = eso.total_kwh("DistrictCooling:Facility","TimeStep")
    os.chdir(root_dir)
    clear_output()
    shutil.rmtree(os.path.join(base_path,f"{groupname}")) 


In [ ]:
result_df.to_csv(os.path.join(root_dir,"results.csv"))
df.to_csv(os.path.join(root_dir,"all_parameters.csv"))